# Neural Networks in PyTorch — Clean Solution

Habitat Threat Detector + revision problems.

**Key concepts:** nn.Sequential, DataLoader, BCELoss, Adam, binary classification, evaluation with confusion matrix.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Revision Problem 1: Simple Sequential Network

In [ ]:
model = nn.Sequential(
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
    nn.Sigmoid()
)
print(model)

## Revision Problem 2: NumPy to Tensor

In [ ]:
data = np.array([1, 2, 3, 4, 5])
tensor_data = torch.tensor(data, dtype=torch.float32)
print(tensor_data)

## Revision Problem 3: CrossEntropyLoss

In [ ]:
logits = torch.tensor([[1.0, 2.0, 0.5], [1.5, 0.2, 1.7], [0.4, 0.8, 2.1]], dtype=torch.float32)
labels = torch.tensor([1, 2, 0], dtype=torch.long)
loss_fun = nn.CrossEntropyLoss()
loss = loss_fun(logits, labels)
print(f"Cross-Entropy Loss: {loss.item():.4f}")

## Main Task: Habitat Threat Detector

In [ ]:
# Load data (assumes CSV is in the same folder or adjust path)
df = pd.read_csv("habitat_images_codebasics_DL.csv")
X = df.drop(columns=['Threat Label']).values
y = df['Threat Label'].values

# Min-Max normalize
X = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))

# To tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

train_data = TensorDataset(X_train, y_train)
test_data = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader = DataLoader(test_data, batch_size=16, shuffle=False)

In [ ]:
class ThreatDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

model = ThreatDetector().to(device)
print(model)

In [ ]:
BCE_loss = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

def train_model(model, train_loader, loss_fn, optimizer, epochs=20):
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(train_loader):.4f}")

train_model(model, train_loader, BCE_loss, optimizer, epochs=20)

In [ ]:
def evaluate_model(model, test_loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_pred = model(X_batch)
            all_preds.extend(y_pred.cpu().numpy())
            all_labels.extend(y_batch.numpy())
    y_pred_binary = (np.array(all_preds) > 0.5).astype(int).flatten()
    accuracy = accuracy_score(all_labels, y_pred_binary)
    print(f"Accuracy: {accuracy:.2f}")
    cm = confusion_matrix(all_labels, y_pred_binary)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.show()

evaluate_model(model, test_loader)